# AIO003 – Ensemble Model for Phoenix Risk Forecasting

**Task:** AIO003 – Ensemble Model  
**Author:** Tarun Kumar Atla  
**Base task:** AI013 – Time Series Forecasting  

---

## Objective
AI013 established a single LSTM model to forecast daily Fire Radiative Power (FRP in MW) from Australian wildfire data.  
This notebook extends that by training three independent models — **LSTM**, **Gradient Boosting**, and **Random Forest** — and combining them into a **weighted ensemble** where each model's contribution is proportional to its accuracy on the test set.

All three models are evaluated on the **same test window** (date-aligned) so comparisons are fair.

---

## Dataset
- `wildfire_multi_region_dataset.csv` — Australian satellite wildfire detections, aggregated to daily level  
- **Target:** `frp_mw` (Fire Radiative Power, MW) — direct measure of fire intensity  
- **Features:** `temp_max_c`, `wind_max_kmh`, `precip_mm`, `humidity_pct`, `brightness_k`, `event_count`

In [ ]:
# ── 1. Imports & Config ──────────────────────────────────────────────────────
import os, sys, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

# ── Resolve AI013 src — works in VS Code, JupyterLab, and classic Jupyter ──
def _find_ai013_src():
    """Walk up from cwd until we find ai013-forecasting/src."""
    current = os.path.abspath(".")
    for _ in range(10):
        for sub in ["ai013-forecasting", os.path.join("models", "ai013-forecasting")]:
            candidate = os.path.join(current, sub, "src")
            if os.path.isdir(candidate):
                return os.path.abspath(candidate), os.path.abspath(os.path.join(current, sub))
        current = os.path.dirname(current)
    return None, None

try:
    _nb_path     = __vsc_ipynb_file__
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(_nb_path))
    ROOT         = os.path.dirname(NOTEBOOK_DIR)
    AI013_ROOT   = os.path.abspath(os.path.join(ROOT, "..", "..", "ai013-forecasting"))
    AI013_SRC    = os.path.join(AI013_ROOT, "src")
    if not os.path.isdir(AI013_SRC):
        raise FileNotFoundError
except Exception:
    AI013_SRC, AI013_ROOT = _find_ai013_src()
    if AI013_SRC is None:
        raise RuntimeError("Cannot locate ai013-forecasting/src — run from anywhere inside the Phoenix repo.")
    ROOT = os.path.abspath(os.path.join(AI013_SRC, "..", "..", "..", "time-series", "aio003-ensemble"))

DATA_PATH   = os.path.join(AI013_ROOT, "data", "wildfire_multi_region_dataset.csv")
PLOTS_DIR   = os.path.join(ROOT, "plots")
OUTPUTS_DIR = os.path.join(ROOT, "outputs")
os.makedirs(PLOTS_DIR,   exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

sys.path.insert(0, AI013_SRC)

WINDOW      = 10
TRAIN_SPLIT = 0.80
SEED        = 42
DEVICE      = "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Setup complete")
print(f"AI013 src : {AI013_SRC}")
print(f"Data      : {DATA_PATH}")
print(f"Plots     : {PLOTS_DIR}")


---
## 2. Data Loading & Preprocessing

In [ ]:
# ── 2. Load data ──────────────────────────────────────────────────────────────
from dataset_builder import build_dataset, FEATURE_COLS, TARGET_COL, DATE_COL

daily = build_dataset(DATA_PATH)
print(f'\nFeatures : {FEATURE_COLS}')
print(f'Target   : {TARGET_COL}')
daily[['acq_date','frp_mw'] + FEATURE_COLS].head(3)

In [ ]:
# ── 3. Scale & build LSTM sequences ──────────────────────────────────────────
feat_scaler   = MinMaxScaler()
target_scaler = MinMaxScaler()

X_all = feat_scaler.fit_transform(daily[FEATURE_COLS].values)
y_all = target_scaler.fit_transform(daily[[TARGET_COL]].values)

def make_sequences(X, y, window):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = make_sequences(X_all, y_all, WINDOW)
split = int(len(X_seq) * TRAIN_SPLIT)   # index in X_seq

# date of first test sample (both LSTM and tree models will start here)
test_start_date = daily[DATE_COL].iloc[split + WINDOW]
print(f'Train/test boundary : {test_start_date.date()}')
print(f'LSTM train samples  : {split:,}')
print(f'LSTM test  samples  : {len(X_seq)-split:,}')

X_train_seq = torch.tensor(X_seq[:split], dtype=torch.float32)
y_train_seq = torch.tensor(y_seq[:split], dtype=torch.float32)
X_test_seq  = torch.tensor(X_seq[split:], dtype=torch.float32)
y_test_seq  = torch.tensor(y_seq[split:], dtype=torch.float32)

In [ ]:
# ── 4. Build tree features with lag columns (date-aligned to LSTM test window) ─
LAG_DAYS = [1, 2, 3, 7]
df_tree = daily.copy()
for lag in LAG_DAYS:
    df_tree[f'frp_lag_{lag}'] = df_tree[TARGET_COL].shift(lag)
df_tree = df_tree.dropna().reset_index(drop=True)

TREE_FEATS = FEATURE_COLS + [f'frp_lag_{l}' for l in LAG_DAYS]

# split exactly on the same date as LSTM
tree_test_mask  = df_tree[DATE_COL] >= test_start_date
tree_train_mask = ~tree_test_mask

X_tr = df_tree.loc[tree_train_mask, TREE_FEATS].values
y_tr = df_tree.loc[tree_train_mask, TARGET_COL].values
X_te = df_tree.loc[tree_test_mask,  TREE_FEATS].values
y_te = df_tree.loc[tree_test_mask,  TARGET_COL].values  # original scale (MW)

print(f'Tree train: {len(X_tr):,}  |  Tree test: {len(X_te):,}')
print(f'Tree features ({len(TREE_FEATS)}): {TREE_FEATS}')

---
## 3. Model 1 — LSTM

In [ ]:
# ── LSTM (same architecture as AI013) ────────────────────────────────────────
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=64, dropout=0.2, num_layers=1):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

lstm_model = LSTMForecaster(len(FEATURE_COLS)).to(DEVICE)
loader     = DataLoader(TensorDataset(X_train_seq, y_train_seq), batch_size=32, shuffle=True)
loss_fn    = nn.MSELoss()
opt        = torch.optim.Adam(lstm_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=20, factor=0.5)

train_losses, val_losses = [], []
best_val, best_w = float('inf'), None

for epoch in range(1, 201):
    lstm_model.train()
    bl = []
    for xb, yb in loader:
        opt.zero_grad()
        loss = loss_fn(lstm_model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        opt.step()
        bl.append(loss.item())
    t_loss = np.mean(bl)

    lstm_model.eval()
    with torch.no_grad():
        v_loss = loss_fn(lstm_model(X_test_seq), y_test_seq).item()

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    scheduler.step(v_loss)

    if v_loss < best_val:
        best_val = v_loss
        best_w   = {k: v.clone() for k, v in lstm_model.state_dict().items()}

    if epoch % 50 == 0:
        print(f'epoch {epoch:3d}  train: {t_loss:.5f}  val: {v_loss:.5f}')

lstm_model.load_state_dict(best_w)
print(f'\nLSTM done — best val loss: {best_val:.5f}')

---
## 4. Model 2 — Gradient Boosting

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05,
    max_depth=4, subsample=0.8, random_state=SEED
)
gb_model.fit(X_tr, y_tr)
print('Gradient Boosting trained')

---
## 5. Model 3 — Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=8,
    min_samples_leaf=2, random_state=SEED, n_jobs=-1
)
rf_model.fit(X_tr, y_tr)
print('Random Forest trained')

---
## 6. Align Predictions to the Same Test Window

In [ ]:
# ── LSTM predictions (inverse-scaled back to MW) ──────────────────────────────
lstm_model.eval()
with torch.no_grad():
    lstm_pred_sc = lstm_model(X_test_seq).numpy()
lstm_pred = target_scaler.inverse_transform(lstm_pred_sc).flatten()
lstm_true = target_scaler.inverse_transform(y_test_seq.numpy()).flatten()

# ── Tree predictions (already in MW — no scaling needed) ─────────────────────
gb_pred = gb_model.predict(X_te)
rf_pred = rf_model.predict(X_te)

# ── Align to common length (both should already match on test_start_date) ─────
n = min(len(lstm_pred), len(gb_pred))
y_true  = lstm_true[:n]
p_lstm  = lstm_pred[:n]
p_gb    = gb_pred[:n]
p_rf    = rf_pred[:n]

# ── Baseline: rolling 7-day mean ──────────────────────────────────────────────
test_start_idx = split + WINDOW
p_baseline = np.array([
    daily[TARGET_COL].values[test_start_idx + i - 7 : test_start_idx + i].mean()
    for i in range(n)
])

print(f'Aligned test samples : {n}')
print(f'Test date range      : {daily[DATE_COL].iloc[test_start_idx].date()} → '
      f'{daily[DATE_COL].iloc[test_start_idx + n - 1].date()}')

---
## 7. Individual Model Evaluation

In [ ]:
def compute_metrics(y_true, y_pred, label):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100
    return {'Model': label,
            'MAE':  round(mae, 3),
            'RMSE': round(rmse, 3),
            'R2':   round(r2, 4),
            'MAPE': round(mape, 2)}

m_base = compute_metrics(y_true, p_baseline, 'Baseline (Rolling Mean)')
m_lstm = compute_metrics(y_true, p_lstm,     'LSTM')
m_gb   = compute_metrics(y_true, p_gb,       'Gradient Boosting')
m_rf   = compute_metrics(y_true, p_rf,       'Random Forest')

individual_df = pd.DataFrame([m_base, m_lstm, m_gb, m_rf])
print('── Individual Model Results ─────────────────────────────────')
print(individual_df.to_string(index=False))

---
## 8. Weighted Ensemble

**Weight = (1 / MAE)² — squared inverse MAE, normalised to sum to 1**

Squaring the inverse penalises poor performers more aggressively than a simple 1/MAE.
A model with half the error gets four times the weight — so the ensemble stays close to the best-performing model rather than being dragged toward weaker ones.

In [ ]:
# ── Squared inverse MAE weights ───────────────────────────────────────────────
inv_sq  = np.array([1/m_lstm['MAE']**2, 1/m_gb['MAE']**2, 1/m_rf['MAE']**2])
weights = inv_sq / inv_sq.sum()

print('── Ensemble Weights (squared inverse MAE) ───────────────────')
print(f'  LSTM              : {weights[0]:.3f}  ({weights[0]*100:.1f}%)')
print(f'  Gradient Boosting : {weights[1]:.3f}  ({weights[1]*100:.1f}%)')
print(f'  Random Forest     : {weights[2]:.3f}  ({weights[2]*100:.1f}%)')

p_ensemble = weights[0]*p_lstm + weights[1]*p_gb + weights[2]*p_rf

m_ens = compute_metrics(y_true, p_ensemble, 'Ensemble (Weighted Avg)')

all_df = pd.DataFrame([m_base, m_lstm, m_gb, m_rf, m_ens])
print('\n── Full Comparison ──────────────────────────────────────────')
print(all_df.to_string(index=False))

best_single_mae = min(m_lstm['MAE'], m_gb['MAE'], m_rf['MAE'])
improvement = (best_single_mae - m_ens['MAE']) / best_single_mae * 100
print(f'\nEnsemble MAE improvement over best single model: {improvement:.2f}%')

---
## 9. Visualisations

In [ ]:
# ── Plot 1: Predictions vs Actual ─────────────────────────────────────────────
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('ggplot')

fig, ax = plt.subplots(figsize=(16, 5))
x = range(n)

ax.plot(x, y_true,      color='#2c3e50', lw=2.2,  label='Actual',             zorder=5)
ax.plot(x, p_ensemble,  color='#e74c3c', lw=2.0,  label='Ensemble',            zorder=4)
ax.plot(x, p_lstm,      color='#3498db', lw=1.3,  label='LSTM',      ls='--',  alpha=0.85)
ax.plot(x, p_gb,        color='#27ae60', lw=1.3,  label='Grad. Boost',ls='--', alpha=0.85)
ax.plot(x, p_rf,        color='#9b59b6', lw=1.3,  label='Rand. Forest',ls='--',alpha=0.85)
ax.plot(x, p_baseline,  color='#e67e22', lw=1.2,  label='Baseline',  ls=':',  alpha=0.8)

ax.set_title('Ensemble vs Individual Models — Fire Radiative Power (MW)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Test Sample Index')
ax.set_ylabel('FRP (MW)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'ensemble_predictions.png'), dpi=150)
plt.show()

In [ ]:
# ── Plot 2: Error Metric Bar Charts ───────────────────────────────────────────
model_labels = [r['Model'] for r in [m_base, m_lstm, m_gb, m_rf, m_ens]]
mae_vals     = [r['MAE']   for r in [m_base, m_lstm, m_gb, m_rf, m_ens]]
rmse_vals    = [r['RMSE']  for r in [m_base, m_lstm, m_gb, m_rf, m_ens]]
colours      = ['#e67e22','#3498db','#27ae60','#9b59b6','#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, c in enumerate(colours):
    axes[0].bar(i, mae_vals[i],  color=c, edgecolor='black', alpha=0.85)
    axes[1].bar(i, rmse_vals[i], color=c, edgecolor='black', alpha=0.85)

for ax, vals, title in zip(axes, [mae_vals, rmse_vals], ['MAE', 'RMSE']):
    ax.set_xticks(range(len(model_labels)))
    ax.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=9)
    ax.set_title(f'{title} — Lower is Better', fontsize=12, fontweight='bold')
    ax.set_ylabel(f'{title} (MW)')
    for i, v in enumerate(vals):
        ax.text(i, v + max(vals)*0.01, f'{v:.1f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Model Comparison — Error Metrics', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'model_comparison_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: R² Comparison ─────────────────────────────────────────────────────
r2_vals = [r['R2'] for r in [m_base, m_lstm, m_gb, m_rf, m_ens]]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(model_labels, r2_vals, color=colours, edgecolor='black', alpha=0.85)
ax.axhline(0, color='red', lw=1.2, ls='--', alpha=0.5, label='R²=0 (predicting mean)')
ax.set_title('R² Score — Higher is Better', fontsize=12, fontweight='bold')
ax.set_ylabel('R² Score')
ax.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=9)
ax.legend(fontsize=9)
for bar, v in zip(bars, r2_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005 if v >= 0 else bar.get_height() - 0.05,
            f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'r2_comparison.png'), dpi=150)
plt.show()

In [ ]:
# ── Plot 4: LSTM Training Loss ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train Loss', color='#2980b9', lw=1.5)
ax.plot(val_losses,   label='Val Loss',   color='#e74c3c', lw=1.5)
ax.set_title('LSTM Training Loss (MSE)', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'lstm_training_loss.png'), dpi=150)
plt.show()

---
## 10. Save Outputs

In [ ]:
# predictions CSV
pd.DataFrame({
    'actual':            y_true,
    'lstm':              p_lstm,
    'gradient_boosting': p_gb,
    'random_forest':     p_rf,
    'baseline':          p_baseline,
    'ensemble':          p_ensemble,
}).to_csv(os.path.join(OUTPUTS_DIR, 'aio003_ensemble_predictions.csv'), index=False)

# metrics CSV
all_df.to_csv(os.path.join(OUTPUTS_DIR, 'aio003_model_metrics.csv'), index=False)

print('── Outputs saved ─────────────────────────────────────────────')
print(f'  outputs/aio003_ensemble_predictions.csv')
print(f'  outputs/aio003_model_metrics.csv')
print(f'  plots/  (4 figures)')

print('\n── Final Results ─────────────────────────────────────────────')
print(all_df.to_string(index=False))

print(f'\nEnsemble MAE improvement over best single model: {improvement:.2f}%')

---
## 11. Summary & Phoenix Integration

### What was built

Three independent models were trained on the same Australian wildfire time series data and evaluated on the same held-out test window:

| Model | Type | What it captures |
|---|---|---|
| LSTM | Recurrent neural network | Temporal dependencies across the 10-day sliding window |
| Gradient Boosting | Tree-based ensemble | Non-linear relationships between weather features and fire intensity |
| Random Forest | Tree-based ensemble | Stable predictions via variance reduction across 300 trees |

Their predictions are combined using a **weighted average**, where each model's weight is the inverse of its MAE — so more accurate models contribute more to the final output.

### Why this improves on AI013
A single LSTM can struggle when fire intensity spikes suddenly or when seasonal patterns shift. By including tree models that approach the same problem from a feature-based perspective, the ensemble is less likely to be badly wrong in any one situation. The worst-case error is bounded by the average of all three, rather than the worst case of one.

### Phoenix integration
The ensemble output (`ensemble` column in `aio003_ensemble_predictions.csv`) is a drop-in replacement for the AI013 single-model output. The Phoenix risk engine receives the same FRP (MW) signal, now from a more robust source. No downstream schema changes are needed.

### Limitations
- Weights are computed on the test set itself — in a production system a separate validation window should be used to set weights, then evaluate on a true holdout
- The ensemble currently covers wildfire data only; extending to cyber threat and disaster mapper datasets would broaden Phoenix risk coverage
- A stacking approach (training a meta-learner on model outputs) could push accuracy further at the cost of added complexity